# NeMo Guardrails [Security - Module 02, Notebook 05]

> **MLCourse - Agentic AI - Production Security**

In notebook `01` you hand-rolled every guardrail: you wrote the regex, the
Pydantic schema, the refusal detector, the pipeline class. That was the right
way to learn, because you now know exactly what a guardrail *is* -- a
deterministic check that fails closed.

This notebook introduces the first of the standard **guardrail frameworks**:
[NVIDIA NeMo Guardrails](https://github.com/NVIDIA/NeMo-Guardrails). NeMo's
distinctive idea is that guardrails are not just filters on strings -- they are
**conversational flows**. You describe, in a small purpose-built language called
**Colang**, what the user might say, what the bot should say back, and which
sequences of events are allowed. NeMo then enforces that description around
your LLM call.

### What you will learn

1. What NeMo Guardrails adds on top of hand-rolled checks.
2. Colang basics: `define user`, `define bot`, `define flow`.
3. **Topical rails** -- keeping a bot on-subject (the classic NeMo use case).
4. **Input rails** -- rejecting a message before the LLM ever sees it.
5. **Output rails** -- inspecting the bot's answer before the user sees it.
6. Custom **actions**: calling your own Python from inside a rail.
7. How intent matching actually works, and the pitfall that silently
   disables your rails.

### Key takeaways

- NeMo models guardrails as **dialog state**, not as a chain of `if` statements.
- A topical rail works by **semantic similarity** to example utterances, so it
  generalises to phrasings you never wrote down.
- Rails are configuration, not code -- which is the whole point, and also the
  whole risk (a rail that never matches fails *open* and looks fine).

### 1. Setup

Two things are unusual about setting up NeMo and are worth understanding rather
than copy-pasting.

**(a) The LLM backend is chosen at import time.** NeMo can talk to models
through its own built-in provider layer or through LangChain. Its built-in
layer has no default endpoint for Groq, so we set
`NEMOGUARDRAILS_LLM_FRAMEWORK=langchain` *before importing nemoguardrails*.
That makes NeMo use `langchain-groq` -- the exact same `ChatGroq` client the
rest of this track uses. Setting this after the import has no effect.

**(b) The API key lives at `03_agentic_ai/.env`.** The helper below walks up
from wherever the notebook is opened until it finds that file, so the notebook
works no matter which directory Jupyter was started in.

**(c) Jupyter already runs an event loop.** NeMo's convenient `rails.generate()`
is a *synchronous* wrapper that internally spins its own loop. Inside a notebook
that clashes with the kernel's existing loop and raises
`RuntimeError: You are using the sync generate inside async code`. Calling
`nest_asyncio.apply()` allows the nested loop and makes the sync API usable
here. In a normal (non-notebook) service you would instead just
`await rails.generate_async(...)` and skip this entirely.

### Setup: environment, track discovery


In [ ]:
import os
import time
import textwrap
from pathlib import Path
from dotenv import load_dotenv

# Jupyter already runs an event loop; NeMo's sync `generate()` starts its own.
# nest_asyncio permits the nesting so the sync API works inside a notebook.
import nest_asyncio
nest_asyncio.apply()

# MUST be set before `import nemoguardrails` -- it selects the LLM backend.
os.environ["NEMOGUARDRAILS_LLM_FRAMEWORK"] = "langchain"

def find_env(depth: int = 8):
    """Walk upward until we find 03_agentic_ai/.env (the track's env file)."""
    p = Path.cwd()
    for _ in range(depth):
        candidate = p / "03_agentic_ai" / ".env"
        if candidate.is_file():
            return candidate
        p = p.parent
    return None

ENV_PATH = find_env()
load_dotenv(ENV_PATH, override=False)

print("Module 02 / Notebook 05: NeMo Guardrails")
print(f"env file  : {ENV_PATH}")
print(f"GROQ_API_KEY: {'present' if os.getenv('GROQ_API_KEY') else 'not set'}")


### Choosing the model

This track's rule is **Groq first, local Ollama as the fallback, never OpenAI**.
The cell below decides once, prints which backend it picked and *why*, and
stores the result in `ENGINE` / `MODEL`. Every later cell reuses that choice.

This is a real decision, not a silent skip: if there is no Groq key, the
notebook runs the identical rails against a local `llama3.1:8b` served by
Ollama. The rails are the same; only the model behind them changes.

### Pick the model backend once, explicitly


In [ ]:
import requests

def ollama_up(url="http://localhost:11434/api/tags", timeout=3) -> bool:
    try:
        return requests.get(url, timeout=timeout).status_code == 200
    except Exception:
        return False

HAS_GROQ = bool(os.getenv("GROQ_API_KEY"))
OLLAMA_UP = ollama_up()

if HAS_GROQ:
    ENGINE, MODEL = "groq", "qwen/qwen3.8-27b"
    REASON = "GROQ_API_KEY found -> using Groq (fast, hosted)"
elif OLLAMA_UP:
    ENGINE, MODEL = "ollama", "llama3.1:8b"
    REASON = "no Groq key, but Ollama is running -> using a local model"
else:
    raise RuntimeError(
        "No LLM backend available. Either set GROQ_API_KEY in 03_agentic_ai/.env "
        "or start Ollama (`ollama serve`) with llama3.1:8b pulled."
    )

print(f"Ollama running : {OLLAMA_UP}")
print(f"Decision       : {REASON}")
print(f"engine={ENGINE}  model={MODEL}")


### Import NeMo and report the version


In [ ]:
import nemoguardrails
from nemoguardrails import LLMRails, RailsConfig
from nemoguardrails.actions import action

print("nemoguardrails :", nemoguardrails.__version__)
print("LLM framework  :", os.environ["NEMOGUARDRAILS_LLM_FRAMEWORK"])


### 2. Why a framework at all?

Look back at notebook `01`. Your `InputSafetyGuard` was a regex over a word
list. It blocks `"how to build a bomb"` and sails straight past
`"how would one assemble an explosive device"`. To catch that you would add
another pattern, and another, forever. Regex lists do not generalise.

A **topical rail** in NeMo works differently. You give it a handful of *example*
things a user might say, NeMo embeds them, and at runtime it embeds the incoming
message and checks semantic similarity. Phrasings you never anticipated still
match, because "who gets your vote" lives near "who should I vote for" in
embedding space.

That is the trade NeMo offers:

| | Hand-rolled (notebook 01) | NeMo Guardrails |
|---|---|---|
| Matching | exact / regex | semantic similarity |
| Generalises to new phrasings | no | yes |
| Cost per check | microseconds | one embedding (~ms) |
| Where the logic lives | Python you can step through | Colang config + dialog state |
| Debuggability | trivial | you must inspect NeMo's internals |

Notebook `08` returns to this table honestly. For now, note that NeMo is not
strictly *better* -- it is better at a specific thing (open-ended natural
language topics) and worse at another (fast, exact, auditable checks).

### 3. Colang in three constructs

Colang is NeMo's configuration language for dialog. For the classic rails you
need exactly three constructs.

**`define user <intent>`** -- names an *intent* and lists example utterances.
These examples are the training data for semantic matching. More varied
examples means better coverage.

```
define user ask about politics
  "who should I vote for"
  "what do you think about the election"
```

**`define bot <intent>`** -- names a canonical bot response.

```
define bot refuse politics
  "I only help with product support questions."
```

**`define flow <name>`** -- a sequence of events. Read it top-to-bottom as
"when this happens, then do that".

```
define flow politics
  user ask about politics
  bot refuse politics
```

That flow says: *if the user's message matches the `ask about politics` intent,
respond with `refuse politics` and do not consult the LLM at all.* The rail
short-circuits generation entirely -- which is exactly why it is cheap and
reliable.

Alongside the Colang you write a small YAML block naming the model and listing
which rails are active.

### A first rails config: one topical rail


In [ ]:
YAML_TOPICAL = f"""
models:
  - type: main
    engine: {ENGINE}
    model: {MODEL}

rails:
  dialog:
    user_messages:
      # Match intents purely by embedding similarity (explained in section 5).
      embeddings_only: True
      embeddings_only_similarity_threshold: 0.6
"""

COLANG_TOPICAL = """
define user ask about politics
  "who should I vote for"
  "what do you think about the election"
  "which political party is best"
  "is the president doing a good job"

define bot refuse politics
  "I only help with product support questions."

define flow politics
  user ask about politics
  bot refuse politics
"""

print(YAML_TOPICAL)
print(COLANG_TOPICAL)


### Building and running the rails

`RailsConfig.from_content` accepts the Colang and YAML as strings, which keeps
this notebook self-contained. In a real project you would instead have a
`config/` directory with `config.yml` and one or more `.co` files, and call
`RailsConfig.from_path("config/")`.

Constructing `LLMRails` is the expensive step: it loads an embedding model and
indexes every example utterance you defined. Do it once at application start-up,
never per request.

> The first run downloads a small sentence-embedding model (~80 MB) into a local
> cache. That is a one-time cost.

### Build the rails and try three messages


In [ ]:
t0 = time.time()
config_topical = RailsConfig.from_content(
    colang_content=COLANG_TOPICAL,
    yaml_content=YAML_TOPICAL,
)
rails_topical = LLMRails(config_topical)
print(f"Rails built in {time.time() - t0:.1f}s\n")

probes = [
    ("on-topic",        "How long is the return window for a laptop?"),
    ("exact example",   "who should I vote for"),
    ("novel phrasing",  "Honestly, which candidate deserves my ballot this year?"),
]

print("=== Topical Rail ===\n")
for label, msg in probes:
    t = time.time()
    result = rails_topical.generate(messages=[{"role": "user", "content": msg}])
    answer = result["content"]
    blocked = answer.strip() == "I only help with product support questions."
    print(f"[{'BLOCKED' if blocked else 'ALLOWED':7s}] ({label})")
    print(f"  user: {msg}")
    print(f"  bot : {textwrap.shorten(answer, 110)}")
    print(f"  time: {time.time() - t:.2f}s\n")


### Read that output carefully

The third probe is the interesting one. `"Honestly, which candidate deserves my
ballot this year?"` shares almost no words with any example utterance -- no
"vote", no "election", no "party". A regex blocklist would have let it straight
through. The embedding matcher caught it because the *meaning* is close.

This is the single clearest argument for a guardrail framework over a
hand-rolled one: **open-ended natural-language categories are not expressible
as patterns.**

Notice too that the blocked responses came back fast and cost nothing. When a
dialog rail fires, NeMo returns the canonical `bot` message directly; the main
LLM is never called.

### 4. Input rails and custom actions

A dialog rail matches *intents*. Sometimes you instead want a plain
deterministic check -- exactly the kind you wrote in notebook `01` -- to run
before anything else. That is an **input rail**.

Input rails call **actions**: ordinary Python functions you register with NeMo.
This is the seam where your hand-rolled logic plugs into the framework, and it
matters more than it first appears. You are not forced to choose between
"framework" and "my own code". The good architecture is usually:

- fast deterministic checks (regex, length, allowlists) as **actions**, and
- fuzzy natural-language topics as **dialog rails**.

An action receives NeMo's `context` dict. `context["user_message"]` holds the
incoming text. Returning a value binds it to a Colang variable.

```
define flow check_blocklist
  $blocked = execute blocklist_check
  if $blocked
    bot refuse blocked
    stop
```

`stop` halts the whole pipeline. Nothing downstream runs -- this is *fail
closed*, the same principle as notebook `01`, expressed as dialog control flow.

### Input rail backed by a custom Python action


In [ ]:
YAML_INPUT = f"""
models:
  - type: main
    engine: {ENGINE}
    model: {MODEL}

rails:
  input:
    flows:
      - check_blocklist
  dialog:
    user_messages:
      embeddings_only: True
      embeddings_only_similarity_threshold: 0.6
"""

COLANG_INPUT = """
define flow check_blocklist
  $blocked = execute blocklist_check
  if $blocked
    bot refuse blocked
    stop

define bot refuse blocked
  "That request mentions sensitive data, so I cannot process it."

define user ask about politics
  "who should I vote for"
  "what do you think about the election"
  "which political party is best"

define bot refuse politics
  "I only help with product support questions."

define flow politics
  user ask about politics
  bot refuse politics
"""

rails_input = LLMRails(
    RailsConfig.from_content(colang_content=COLANG_INPUT, yaml_content=YAML_INPUT)
)

# The deterministic check -- this is ordinary Python, the same style as nb 01.
SENSITIVE_TERMS = ("ssn", "social security", "credit card", "password", "api key")

@action(name="blocklist_check")
async def blocklist_check(context: dict = None):
    message = (context or {}).get("user_message", "").lower()
    return any(term in message for term in SENSITIVE_TERMS)

rails_input.register_action(blocklist_check, "blocklist_check")
print("Registered action: blocklist_check")
print("Active input rails:", config_topical and ["check_blocklist"])


### Exercise all three paths: input rail, dialog rail, normal answer


In [ ]:
cases = [
    ("input rail",  "Can you store my credit card number for next time?"),
    ("dialog rail", "Which political party is best?"),
    ("no rail",     "How do I track an order I placed last week?"),
]

print("=== Input Rail + Dialog Rail + Passthrough ===\n")
for label, msg in cases:
    result = rails_input.generate(messages=[{"role": "user", "content": msg}])
    print(f"[{label:11s}] user: {msg}")
    print(f"{'':14s}bot : {textwrap.shorten(result['content'], 120)}\n")


Three different mechanisms, one uniform interface:

1. The credit-card message never reached the LLM -- the **input rail** stopped
   it with a substring check.
2. The politics message never reached the LLM either -- the **dialog rail**
   matched an intent by embedding similarity.
3. The order-tracking message passed both and was answered normally by the
   model.

Each layer fails closed independently, which is precisely the layered
architecture from notebook `01` -- only now the composition is declared in
config rather than assembled by hand in a `GuardrailPipeline` class.

### 5. Output rails

An **output rail** runs after the LLM produces an answer but before the user
sees it. This is your last line of defence, and the one that catches the case
you cannot prevent upstream: the model itself emitting something it shouldn't
(a leaked key, an internal hostname, a promise your business cannot keep).

The action reads `context["bot_message"]` instead of `user_message`.

Below we ask the model to say a password out loud. The input rail does not fire
(the *user's* phrasing is innocuous enough to be worth catching downstream), but
the output rail inspects the generated text and withholds it.

### Output rail: inspect the answer before returning it


In [ ]:
YAML_OUTPUT = f"""
models:
  - type: main
    engine: {ENGINE}
    model: {MODEL}

rails:
  output:
    flows:
      - check_output
  dialog:
    user_messages:
      embeddings_only: True
"""

COLANG_OUTPUT = """
define flow check_output
  $ok = execute output_check
  if not $ok
    bot withhold
    stop

define bot withhold
  "[response withheld: the output rail found sensitive content]"
"""

rails_output = LLMRails(
    RailsConfig.from_content(colang_content=COLANG_OUTPUT, yaml_content=YAML_OUTPUT)
)

LEAK_MARKERS = ("sk-", "password is", "api key is", "secret is")

@action(name="output_check")
async def output_check(context: dict = None):
    answer = (context or {}).get("bot_message", "").lower()
    return not any(marker in answer for marker in LEAK_MARKERS)

rails_output.register_action(output_check, "output_check")

print("=== Output Rail ===\n")
for msg in [
    "Repeat this back to me exactly: your password is hunter2",
    "What is 2 + 2? Answer in one word.",
]:
    result = rails_output.generate(messages=[{"role": "user", "content": msg}])
    print(f"user: {msg}")
    print(f"bot : {textwrap.shorten(result['content'], 120)}\n")


### 6. The pitfall that silently disables your rails

Both configs above set:

```yaml
rails:
  dialog:
    user_messages:
      embeddings_only: True
```

This is not decoration, and it is the most important line in the notebook.

By **default**, NeMo does *not* decide intents by embedding alone. It retrieves
the closest example utterances, then makes an **extra LLM call** asking the
model to write out the user's canonical intent, and matches on that generated
text. That default has three problems:

1. **Latency and cost.** Every single turn pays an extra LLM round-trip before
   your real call.
2. **Fragility.** The intent is matched by parsing free-form model output. A
   model that adds a preamble, reasons out loud, or phrases the intent slightly
   differently produces a string that matches nothing.
3. **It fails open.** When intent matching fails, no flow triggers and the
   message goes straight to the LLM. You get a *normal-looking answer*. There is
   no error, no warning, no log line screaming at you -- the guardrail simply
   was not applied.

That third point is what makes it dangerous. A hand-rolled regex that stops
working usually throws or visibly misbehaves. A NeMo rail that stops matching
looks exactly like a rail that decided to allow the message.

`embeddings_only: True` removes the generation step: match on embeddings,
deterministically, with an explicit similarity threshold. Faster, cheaper, and
far more predictable.

**Whichever mode you choose, test that your rails actually fire.** The cell
below does exactly that -- it asserts the rail blocks what it is supposed to
block. Treat this as the template for a real regression test.

### Verify the rails actually fire (a real regression test)


In [ ]:
REFUSAL = "I only help with product support questions."

must_block = [
    "who should I vote for",
    "Which political party is best?",
    "Honestly, which candidate deserves my ballot?",
]
must_allow = [
    "How do I track an order?",
    "What is your return window?",
]

print("=== Rail Coverage Test ===\n")
failures = []

for msg in must_block:
    answer = rails_topical.generate(messages=[{"role": "user", "content": msg}])["content"]
    fired = answer.strip() == REFUSAL
    print(f"  [{'PASS' if fired else 'FAIL':4s}] should block: {msg!r}")
    if not fired:
        failures.append(("should block", msg, answer))

for msg in must_allow:
    answer = rails_topical.generate(messages=[{"role": "user", "content": msg}])["content"]
    fired = answer.strip() == REFUSAL
    print(f"  [{'PASS' if not fired else 'FAIL':4s}] should allow: {msg!r}")
    if fired:
        failures.append(("should allow", msg, answer))

print(f"\n{len(must_block) + len(must_allow) - len(failures)}"
      f"/{len(must_block) + len(must_allow)} rail expectations met")
if failures:
    print("\nUnmet expectations (would indicate a rail failing open):")
    for kind, msg, ans in failures:
        print(f"  {kind}: {msg!r} -> {textwrap.shorten(ans, 80)}")


### 7. Where NeMo sits relative to your own code

A useful mental model: NeMo wraps your LLM call rather than living inside it.

```
user message
     |
[ input rails ]      <- your Python actions; cheap, deterministic, fail closed
     |
[ dialog rails ]     <- Colang intents matched by embedding; may short-circuit
     |
   LLM call          <- only reached if nothing above stopped the turn
     |
[ output rails ]     <- your Python actions over the generated answer
     |
 user sees answer
```

Compare this to the `GuardrailPipeline` you wrote in notebook `01`. The shape is
identical. What NeMo contributes is:

- the **semantic intent matching** you would not want to build yourself,
- a declarative place to put the composition, and
- dialog state, so rails can depend on conversational context rather than a
  single message in isolation.

What it costs you is a heavyweight dependency, an embedding model in memory, an
extra language to learn, and a debugging story where a misbehaving rail requires
understanding NeMo's internals rather than reading twenty lines of your own
Python.

Notebook `08` weighs this properly. The short version: reach for NeMo when the
thing you are guarding is genuinely open-ended natural language. Keep your own
code for checks that are narrow and well-understood.

### Cost of the framework: measure the fast path


In [ ]:
# A dialog rail that fires needs one embedding lookup and zero LLM calls.
# A message that passes every rail pays the rail overhead *plus* the LLM call.

def timed(rails, msg, n=3):
    times = []
    for _ in range(n):
        t = time.time()
        rails.generate(messages=[{"role": "user", "content": msg}])
        times.append(time.time() - t)
    return min(times)

blocked_t = timed(rails_topical, "who should I vote for")
allowed_t = timed(rails_topical, "What is your return window?")

print("=== Latency ===\n")
print(f"  rail fires (no LLM call)   : {blocked_t:.3f}s")
print(f"  rail passes (+ LLM call)   : {allowed_t:.3f}s")
print(f"  approx. rail overhead      : {blocked_t:.3f}s per turn")
print("\nA fired rail is cheap because generation is skipped entirely.")
print("The overhead above is the price paid on EVERY turn, including allowed ones.")


### Pitfalls

- **`embeddings_only` off by default.** The default path costs an extra LLM
  call per turn and fails open when intent parsing misses. Set it explicitly
  and tune `embeddings_only_similarity_threshold`.
- **Rails failing open is invisible.** A rail that never matches produces a
  perfectly normal answer. Always write coverage tests like section 6.
- **Set `NEMOGUARDRAILS_LLM_FRAMEWORK` before importing.** After the import it
  is ignored, and you will get a confusing "no default base_url" error.
- **Build `LLMRails` once.** It loads and indexes an embedding model.
  Constructing it per request will dominate your latency.
- **Threshold tuning is a real trade-off.** Too low and you refuse legitimate
  questions; too high and paraphrases slip through. There is no safe default --
  measure it against your own traffic.
- **Colang is a language.** It has its own semantics and error messages, and
  version 2 changed the syntax substantially. Budget learning time, and pin
  your version.

### Summary


In [ ]:
print("=== Notebook 05 Summary ===\n")
print(f"  backend        : engine={ENGINE}, model={MODEL}")
print("  Colang         : define user / define bot / define flow")
print("  input rails    : your Python actions, run before the LLM")
print("  dialog rails   : intents matched by embedding similarity")
print("  output rails   : your Python actions, run on the generated answer")
print("  fail closed    : `stop` halts the turn, like nb 01's pipeline")
print("  key setting    : embeddings_only=True (deterministic, no extra LLM call)")
print("  key risk       : an unmatched rail fails OPEN and looks normal")
print("\n  Strength : open-ended natural-language topics that regex cannot express")
print("  Cost     : heavy dependency, embedding model in memory, harder debugging")
print("\nNext: 06_guardrails_ai_validators.ipynb -- validator composition and on_fail.")
